---
## Phase 4 — Cross-Hospital Generalization

We investigate two transfer directions:

- Mayo calibration → FNUSA evaluation
- FNUSA calibration → Mayo evaluation

Thresholds are calculated from the source hospital
and applied without recalibration to the target hospital.

Each hospital's known mains frequency is used
when calculating the spectral features.

This is retrospective site-transfer analysis.

In [23]:
# ============================================================
# NEUROSIGNAL-QC
# PHASE 4 — CELL 1
# Cross-Hospital Transfer Evaluation
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

POWER_FEATURE = "Relative Mains Power (%)"
PROMINENCE_FEATURE = "Local Prominence (dB)"

THRESHOLD_PERCENTILE = 95

HOSPITALS = ["Mayo", "FNUSA"]

# Use the same AND rule established in Phase 3
DETECTION_RULE = "AND"

df = phase3_features_df.copy()

df["Hospital"] = df["Hospital"].astype(str)
df["Category"] = df["Category"].astype(str)

df["Subject"] = (
    df["Subject"]
    .astype(str)
    .str.replace(r"^sub-", "", regex=True)
    .str.zfill(3)
)

df["Patient_ID"] = (
    df["Hospital"] + "_" + df["Subject"]
)

# ------------------------------------------------------------
# 2. Dataset integrity
# ------------------------------------------------------------

assert len(df) == 225, (
    "Unexpected dataset size."
)

assert not df.duplicated(
    ["Hospital", "Store", "Sample Index"]
).any(), "Duplicate recordings detected."

assert np.isfinite(
    df[
        [POWER_FEATURE, PROMINENCE_FEATURE]
    ].to_numpy(dtype=float)
).all(), "Invalid spectral features."

print("=" * 70)
print("NEUROSIGNAL-QC — CROSS-HOSPITAL GENERALIZATION")
print("=" * 70)

print("\nTotal recordings:", len(df))

print(
    "Total participants:",
    df["Patient_ID"].nunique()
)

# ------------------------------------------------------------
# 3. Define cross-hospital experiments
# ------------------------------------------------------------

experiments = [
    ("Mayo", "FNUSA"),
    ("FNUSA", "Mayo")
]

transfer_results = []
transfer_errors = []

# ------------------------------------------------------------
# 4. Evaluate each transfer direction
# ------------------------------------------------------------

for source_hospital, target_hospital in experiments:

    print("\n" + "-" * 70)

    print(
        f"SOURCE: {source_hospital} "
        f"--> TARGET: {target_hospital}"
    )

    source_df = df[
        df["Hospital"] == source_hospital
    ].copy()

    target_df = df[
        df["Hospital"] == target_hospital
    ].copy()

    # --------------------------------------------------------
    # 5. Verify hospital-level independence
    # --------------------------------------------------------

    source_patients = set(
        source_df["Patient_ID"]
    )

    target_patients = set(
        target_df["Patient_ID"]
    )

    overlap = source_patients.intersection(
        target_patients
    )

    assert len(overlap) == 0, (
        "Participant overlap between hospitals!"
    )

    # --------------------------------------------------------
    # 6. Calibrate using SOURCE hospital only
    # --------------------------------------------------------

    source_negative = source_df[
        source_df["Category"] != "Power-line"
    ]

    power_threshold_source = float(
        np.percentile(
            source_negative[POWER_FEATURE],
            THRESHOLD_PERCENTILE
        )
    )

    prominence_threshold_source = float(
        np.percentile(
            source_negative[PROMINENCE_FEATURE],
            THRESHOLD_PERCENTILE
        )
    )

    print(
        "\nSource-calibrated power threshold:",
        round(power_threshold_source, 5),
        "%"
    )

    print(
        "Source-calibrated prominence threshold:",
        round(prominence_threshold_source, 5),
        "dB"
    )

    # --------------------------------------------------------
    # 7. Apply frozen AND rule to TARGET hospital
    # --------------------------------------------------------

    power_flag = (
        target_df[POWER_FEATURE].to_numpy()
        > power_threshold_source
    )

    prominence_flag = (
        target_df[PROMINENCE_FEATURE].to_numpy()
        > prominence_threshold_source
    )

    predicted = (
        power_flag & prominence_flag
    )

    actual = (
        target_df["Category"].to_numpy()
        == "Power-line"
    )

    # --------------------------------------------------------
    # 8. Calculate confusion matrix
    # --------------------------------------------------------

    tp = int(np.sum(predicted & actual))

    fn = int(np.sum(
        (~predicted) & actual
    ))

    fp = int(np.sum(
        predicted & (~actual)
    ))

    tn = int(np.sum(
        (~predicted) & (~actual)
    ))

    # --------------------------------------------------------
    # 9. Pathological annotation analysis
    # --------------------------------------------------------

    pathological = (
        target_df["Category"].to_numpy()
        == "Pathological"
    )

    pathological_total = int(
        np.sum(pathological)
    )

    pathological_flagged = int(
        np.sum(predicted & pathological)
    )

    # --------------------------------------------------------
    # 10. Calculate performance
    # --------------------------------------------------------

    sensitivity = (
        100 * tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        100 * tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    precision = (
        100 * tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    pathological_fpr = (
        100 * pathological_flagged / pathological_total
        if pathological_total > 0
        else np.nan
    )

    # --------------------------------------------------------
    # 11. Store transfer results
    # --------------------------------------------------------

    transfer_results.append({
        "Source": source_hospital,
        "Target": target_hospital,
        "Source Clips": len(source_df),
        "Source Participants": len(source_patients),
        "Target Clips": len(target_df),
        "Target Participants": len(target_patients),
        "Power Threshold (%)":
            power_threshold_source,
        "Prominence Threshold (dB)":
            prominence_threshold_source,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "Sensitivity (%)": sensitivity,
        "Specificity (%)": specificity,
        "Precision (%)": precision,
        "Pathological Total": pathological_total,
        "Pathological Flagged": pathological_flagged,
        "Pathological FPR (%)": pathological_fpr
    })

    # --------------------------------------------------------
    # 12. Collect classification disagreements
    # --------------------------------------------------------

    target_df["Predicted_Powerline"] = predicted
    target_df["Actual_Powerline"] = actual

    incorrect = target_df[
        target_df["Predicted_Powerline"]
        != target_df["Actual_Powerline"]
    ].copy()

    incorrect["Transfer"] = (
        source_hospital + " -> " + target_hospital
    )

    incorrect["Error_Type"] = np.where(
        incorrect["Actual_Powerline"],
        "Missed Power-line",
        "Flagged Other Category"
    )

    transfer_errors.append(incorrect)

    print("\nTarget recordings:", len(target_df))

    print(
        "Target participants:",
        len(target_patients)
    )

    print(
        "Sensitivity:",
        round(sensitivity, 2),
        "%"
    )

    print(
        "Specificity:",
        round(specificity, 2),
        "%"
    )

    print(
        "Pathological recordings flagged:",
        pathological_flagged,
        "/",
        pathological_total
    )

# ============================================================
# 13. Cross-hospital results
# ============================================================

cross_hospital_results_df = pd.DataFrame(
    transfer_results
)

print("\n" + "=" * 70)
print("CROSS-HOSPITAL TRANSFER RESULTS")
print("=" * 70)

display(
    cross_hospital_results_df.round(4)
)

# ============================================================
# 14. Misclassified recordings
# ============================================================

cross_hospital_errors_df = pd.concat(
    transfer_errors,
    ignore_index=True
)

print("\n" + "=" * 70)
print("CROSS-HOSPITAL CLASSIFICATION DISAGREEMENTS")
print("=" * 70)

if len(cross_hospital_errors_df) > 0:

    display(
        cross_hospital_errors_df[
            [
                "Transfer",
                "Subject",
                "Category",
                "Error_Type",
                POWER_FEATURE,
                PROMINENCE_FEATURE
            ]
        ].round(4)
    )

else:

    print(
        "No classification disagreements "
        "in either transfer direction."
    )

# ============================================================
# 15. Final checks
# ============================================================

print("\n" + "=" * 70)
print("CROSS-HOSPITAL EVALUATION COMPLETE")
print("=" * 70)

print("Detection rule:", DETECTION_RULE)

print(
    "Threshold calibration percentile:",
    THRESHOLD_PERCENTILE
)

print(
    "Destination hospital recordings used "
    "for threshold calibration: 0"
)

print(
    "\nThis is retrospective site-transfer analysis, "
    "not independent external clinical validation."
)

NEUROSIGNAL-QC — CROSS-HOSPITAL GENERALIZATION

Total recordings: 225
Total participants: 38

----------------------------------------------------------------------
SOURCE: Mayo --> TARGET: FNUSA

Source-calibrated power threshold: 1.38848 %
Source-calibrated prominence threshold: 24.63914 dB

Target recordings: 105
Target participants: 14
Sensitivity: 100.0 %
Specificity: 100.0 %
Pathological recordings flagged: 0 / 30

----------------------------------------------------------------------
SOURCE: FNUSA --> TARGET: Mayo

Source-calibrated power threshold: 1.07654 %
Source-calibrated prominence threshold: 10.34513 dB

Target recordings: 120
Target participants: 24
Sensitivity: 90.0 %
Specificity: 95.56 %
Pathological recordings flagged: 0 / 30

CROSS-HOSPITAL TRANSFER RESULTS


,Source,Target,Source Clips,Source Participants,Target Clips,Target Participants,Power Threshold (%),Prominence Threshold (dB),TP,FN,FP,TN,Sensitivity (%),Specificity (%),Precision (%),Pathological Total,Pathological Flagged,Pathological FPR (%)
0,Mayo,FNUSA,120,24,105,14,1.3885,24.6391,15,0,0,90,100.0,100.0000,100.0000,30,0,0.0
1,FNUSA,Mayo,105,14,120,24,1.0765,10.3451,27,3,4,86,90.0,95.5556,87.0968,30,0,0.0



CROSS-HOSPITAL CLASSIFICATION DISAGREEMENTS


,Transfer,Subject,Category,Error_Type,Relative Mains Power (%),Local Prominence (dB)
0,FNUSA -> Mayo,005,Physiological,Flagged Other Category,1.5463,18.6481
1,FNUSA -> Mayo,020,Artifact,Flagged Other Category,1.9026,26.3962
2,FNUSA -> Mayo,000,Artifact,Flagged Other Category,4.6254,11.1789
3,FNUSA -> Mayo,020,Artifact,Flagged Other Category,1.5667,20.5051
4,FNUSA -> Mayo,016,Power-line,Missed Power-line,0.0652,14.6892
5,FNUSA -> Mayo,012,Power-line,Missed Power-line,0.2769,30.5654
6,FNUSA -> Mayo,012,Power-line,Missed Power-line,0.8338,32.0281



CROSS-HOSPITAL EVALUATION COMPLETE
Detection rule: AND
Threshold calibration percentile: 95
Destination hospital recordings used for threshold calibration: 0

This is retrospective site-transfer analysis, not independent external clinical validation.


### Statistical Uncertainty and Participant-Level Analysis

We calculate exact binomial confidence intervals
for recording-level performance.

Because recordings from the same participant
may be correlated, these intervals do not
fully account for participant clustering.

We therefore also examine classification errors
and complete-detection outcomes by participant.

In [24]:
# ============================================================
# NEUROSIGNAL-QC
# PHASE 4 — CELL 2
# Statistical Uncertainty and Participant-Level Error Audit
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import beta
from IPython.display import display

# ------------------------------------------------------------
# 1. Exact binomial confidence interval
# ------------------------------------------------------------

def exact_binomial_ci(successes, total, alpha=0.05):

    if total == 0:
        return np.nan, np.nan

    lower = (
        0.0
        if successes == 0
        else beta.ppf(
            alpha / 2,
            successes,
            total - successes + 1
        )
    )

    upper = (
        1.0
        if successes == total
        else beta.ppf(
            1 - alpha / 2,
            successes + 1,
            total - successes
        )
    )

    return 100 * lower, 100 * upper


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

POWER_FEATURE = "Relative Mains Power (%)"
PROMINENCE_FEATURE = "Local Prominence (dB)"

uncertainty_records = []
patient_records = []

print("=" * 70)
print("NEUROSIGNAL-QC — STATISTICAL UNCERTAINTY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 3. Reconstruct the original cross-hospital predictions
# ------------------------------------------------------------

for _, result in cross_hospital_results_df.iterrows():

    source = result["Source"]
    target = result["Target"]

    power_threshold = float(
        result["Power Threshold (%)"]
    )

    prominence_threshold = float(
        result["Prominence Threshold (dB)"]
    )

    target_df = phase3_features_df[
        phase3_features_df["Hospital"] == target
    ].copy()

    # Use the same frozen AND rule
    target_df["Predicted"] = (
        (target_df[POWER_FEATURE] > power_threshold)
        &
        (target_df[PROMINENCE_FEATURE] > prominence_threshold)
    )

    target_df["Actual"] = (
        target_df["Category"] == "Power-line"
    )

    target_df["Patient_ID"] = (
        target_df["Hospital"].astype(str)
        + "_"
        + target_df["Subject"].astype(str)
          .str.replace(r"^sub-", "", regex=True)
          .str.zfill(3)
    )

    # --------------------------------------------------------
    # 4. Verify original classification results
    # --------------------------------------------------------

    tp = int((
        target_df["Predicted"]
        &
        target_df["Actual"]
    ).sum())

    fn = int((
        ~target_df["Predicted"]
        &
        target_df["Actual"]
    ).sum())

    fp = int((
        target_df["Predicted"]
        &
        ~target_df["Actual"]
    ).sum())

    tn = int((
        ~target_df["Predicted"]
        &
        ~target_df["Actual"]
    ).sum())

    assert tp == int(result["TP"])
    assert fn == int(result["FN"])
    assert fp == int(result["FP"])
    assert tn == int(result["TN"])

    # --------------------------------------------------------
    # 5. Calculate exact confidence intervals
    # --------------------------------------------------------

    pathological = target_df[
        target_df["Category"] == "Pathological"
    ]

    pathology_flagged = int(
        pathological["Predicted"].sum()
    )

    measurements = [
        (
            "Power-line sensitivity",
            tp,
            tp + fn
        ),
        (
            "Non-power-line specificity",
            tn,
            tn + fp
        ),
        (
            "Pathological flag rate",
            pathology_flagged,
            len(pathological)
        )
    ]

    for metric_name, successes, total in measurements:

        lower, upper = exact_binomial_ci(
            successes,
            total
        )

        uncertainty_records.append({
            "Transfer": f"{source} -> {target}",
            "Metric": metric_name,
            "Numerator": successes,
            "Denominator": total,
            "Estimate (%)": (
                100 * successes / total
                if total else np.nan
            ),
            "95% CI Lower": lower,
            "95% CI Upper": upper
        })

    # --------------------------------------------------------
    # 6. Create participant-level error indicators
    # --------------------------------------------------------

    target_df["TP_Flag"] = (
        target_df["Actual"]
        &
        target_df["Predicted"]
    ).astype(int)

    target_df["FN_Flag"] = (
        target_df["Actual"]
        &
        ~target_df["Predicted"]
    ).astype(int)

    target_df["FP_Flag"] = (
        ~target_df["Actual"]
        &
        target_df["Predicted"]
    ).astype(int)

    target_df["Pathology_Flag"] = (
        (target_df["Category"] == "Pathological")
        &
        target_df["Predicted"]
    ).astype(int)

    # --------------------------------------------------------
    # 7. Aggregate errors by participant
    # --------------------------------------------------------

    patient_summary = (
        target_df
        .groupby("Patient_ID", observed=True)
        .agg(
            Clips=("Actual", "size"),
            Powerline_Clips=("Actual", "sum"),
            Powerline_Detected=("TP_Flag", "sum"),
            Powerline_Missed=("FN_Flag", "sum"),
            False_Alarms=("FP_Flag", "sum"),
            Pathological_Flagged=("Pathology_Flag", "sum")
        )
        .reset_index()
    )

    patient_summary["Transfer"] = (
        f"{source} -> {target}"
    )

    patient_records.append(patient_summary)

# ============================================================
# 8. Confidence interval results
# ============================================================

uncertainty_df = pd.DataFrame(
    uncertainty_records
)

print("\n" + "=" * 70)
print("95% EXACT BINOMIAL CONFIDENCE INTERVALS")
print("=" * 70)

display(
    uncertainty_df.round(2)
)

# ============================================================
# 9. Participant-level summaries
# ============================================================

patient_transfer_df = pd.concat(
    patient_records,
    ignore_index=True
)

participant_summary_records = []

for transfer, group in patient_transfer_df.groupby(
    "Transfer",
    observed=True
):

    powerline_patients = group[
        group["Powerline_Clips"] > 0
    ]

    participants_with_misses = int(
        (group["Powerline_Missed"] > 0).sum()
    )

    participants_with_false_alarms = int(
        (group["False_Alarms"] > 0).sum()
    )

    fully_detected = int(
        (
            powerline_patients["Powerline_Missed"] == 0
        ).sum()
    )

    participant_summary_records.append({
        "Transfer": transfer,
        "Target Participants": len(group),
        "Participants With Power-line Clips":
            len(powerline_patients),
        "Power-line Participants Fully Detected":
            fully_detected,
        "Participants With Missed Power-line":
            participants_with_misses,
        "Participants With False Alarms":
            participants_with_false_alarms
    })

participant_summary_df = pd.DataFrame(
    participant_summary_records
)

print("\n" + "=" * 70)
print("PARTICIPANT-LEVEL PERFORMANCE SUMMARY")
print("=" * 70)

display(participant_summary_df)

# ============================================================
# 10. Identify participants contributing errors
# ============================================================

affected_patients_df = patient_transfer_df[
    (patient_transfer_df["Powerline_Missed"] > 0)
    |
    (patient_transfer_df["False_Alarms"] > 0)
].copy()

print("\n" + "=" * 70)
print("PARTICIPANTS CONTRIBUTING CLASSIFICATION ERRORS")
print("=" * 70)

if len(affected_patients_df) > 0:

    display(
        affected_patients_df[
            [
                "Transfer",
                "Patient_ID",
                "Clips",
                "Powerline_Clips",
                "Powerline_Detected",
                "Powerline_Missed",
                "False_Alarms",
                "Pathological_Flagged"
            ]
        ].sort_values(
            ["Transfer", "Patient_ID"]
        )
    )

else:

    print(
        "No participant contributed "
        "classification errors."
    )

# ============================================================
# 11. Final verification
# ============================================================

print("\n" + "=" * 70)
print("UNCERTAINTY AUDIT COMPLETE")
print("=" * 70)

print(
    "Total target participants examined:",
    patient_transfer_df["Patient_ID"].nunique()
)

print(
    "Total classification disagreements:",
    int(
        patient_transfer_df["Powerline_Missed"].sum()
        +
        patient_transfer_df["False_Alarms"].sum()
    )
)

print("\nNo thresholds were modified.")

print(
    "Confidence intervals assume independent clips."
)

print(
    "Participant-level analysis is descriptive."
)

print(
    "Results are exploratory, not clinical validation."
)

NEUROSIGNAL-QC — STATISTICAL UNCERTAINTY AUDIT

95% EXACT BINOMIAL CONFIDENCE INTERVALS


,Transfer,Metric,Numerator,Denominator,Estimate (%),95% CI Lower,95% CI Upper
0,Mayo -> FNUSA,Power-line sensitivity,15,15,100.00,78.20,100.00
1,Mayo -> FNUSA,Non-power-line specificity,90,90,100.00,95.98,100.00
2,Mayo -> FNUSA,Pathological flag rate,0,30,0.00,0.00,11.57
3,FNUSA -> Mayo,Power-line sensitivity,27,30,90.00,73.47,97.89
4,FNUSA -> Mayo,Non-power-line specificity,86,90,95.56,89.01,98.78
5,FNUSA -> Mayo,Pathological flag rate,0,30,0.00,0.00,11.57



PARTICIPANT-LEVEL PERFORMANCE SUMMARY


,Transfer,Target Participants,Participants With Power-line Clips,Power-line Participants Fully Detected,Participants With Missed Power-line,Participants With False Alarms
0,FNUSA -> Mayo,24,8,6,2,3
1,Mayo -> FNUSA,14,3,3,0,0



PARTICIPANTS CONTRIBUTING CLASSIFICATION ERRORS


,Transfer,Patient_ID,Clips,Powerline_Clips,Powerline_Detected,Powerline_Missed,False_Alarms,Pathological_Flagged
14,FNUSA -> Mayo,Mayo_000,5,0,0,0,1,0
19,FNUSA -> Mayo,Mayo_005,4,0,0,0,1,0
26,FNUSA -> Mayo,Mayo_012,4,4,2,2,0,0
30,FNUSA -> Mayo,Mayo_016,8,4,3,1,0,0
34,FNUSA -> Mayo,Mayo_020,4,0,0,0,2,0



UNCERTAINTY AUDIT COMPLETE
Total target participants examined: 38
Total classification disagreements: 7

No thresholds were modified.
Confidence intervals assume independent clips.
Participant-level analysis is descriptive.
Results are exploratory, not clinical validation.


In [25]:
# ============================================================
# NEUROSIGNAL-QC
# PHASE 4 — CELL 3
# Participant-Level Reliability and Confidence Intervals
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import beta
from IPython.display import display

# ------------------------------------------------------------
# 1. Exact binomial confidence interval
# ------------------------------------------------------------

def participant_binomial_ci(events, total, alpha=0.05):

    if total == 0:
        return np.nan, np.nan

    lower = (
        0.0
        if events == 0
        else beta.ppf(
            alpha / 2,
            events,
            total - events + 1
        )
    )

    upper = (
        1.0
        if events == total
        else beta.ppf(
            1 - alpha / 2,
            events + 1,
            total - events
        )
    )

    return 100 * lower, 100 * upper


# ------------------------------------------------------------
# 2. Reuse participant-level results from Phase 4 Cell 2
# ------------------------------------------------------------

patient_df = patient_transfer_df.copy()

required_columns = [
    "Transfer",
    "Patient_ID",
    "Clips",
    "Powerline_Clips",
    "Powerline_Missed",
    "False_Alarms",
    "Pathological_Flagged"
]

assert all(
    col in patient_df.columns
    for col in required_columns
), "Missing participant-level analysis columns."

assert not patient_df.duplicated(
    ["Transfer", "Patient_ID"]
).any(), "Duplicate participant summaries."

print("=" * 70)
print("NEUROSIGNAL-QC — PARTICIPANT-LEVEL RELIABILITY")
print("=" * 70)

# ------------------------------------------------------------
# 3. Identify participants with pathological recordings
# ------------------------------------------------------------

feature_df = phase3_features_df.copy()

feature_df["Patient_ID"] = (
    feature_df["Hospital"].astype(str)
    + "_"
    + feature_df["Subject"]
        .astype(str)
        .str.replace(r"^sub-", "", regex=True)
        .str.zfill(3)
)

# ------------------------------------------------------------
# 4. Evaluate each transfer direction
# ------------------------------------------------------------

participant_metric_records = []

for transfer, group in patient_df.groupby(
    "Transfer",
    observed=True
):

    source, target = transfer.split(" -> ")

    target_features = feature_df[
        feature_df["Hospital"] == target
    ]

    # --------------------------------------------------------
    # A. Complete detection of power-line recordings
    # --------------------------------------------------------

    powerline_patients = group[
        group["Powerline_Clips"] > 0
    ]

    complete_detection = int(
        (powerline_patients["Powerline_Missed"] == 0).sum()
    )

    total_powerline_patients = len(powerline_patients)

    # --------------------------------------------------------
    # B. Participants experiencing at least one false alarm
    # --------------------------------------------------------

    negative_patients = group[
        group["Clips"] > group["Powerline_Clips"]
    ]

    participants_with_false_alarms = int(
        (negative_patients["False_Alarms"] > 0).sum()
    )

    total_negative_patients = len(negative_patients)

    # --------------------------------------------------------
    # C. Participants with pathological recordings flagged
    # --------------------------------------------------------

    pathology_patient_ids = set(
        target_features.loc[
            target_features["Category"] == "Pathological",
            "Patient_ID"
        ]
    )

    pathology_patients = group[
        group["Patient_ID"].isin(pathology_patient_ids)
    ]

    participants_with_pathology_flags = int(
        (pathology_patients["Pathological_Flagged"] > 0).sum()
    )

    total_pathology_patients = len(pathology_patients)

    # --------------------------------------------------------
    # 5. Calculate participant-level confidence intervals
    # --------------------------------------------------------

    measurements = [
        (
            "All Power-line Clips Detected",
            complete_detection,
            total_powerline_patients
        ),
        (
            "Any False Alarm",
            participants_with_false_alarms,
            total_negative_patients
        ),
        (
            "Any Pathological Flag",
            participants_with_pathology_flags,
            total_pathology_patients
        )
    ]

    for metric, count, total in measurements:

        ci_lower, ci_upper = participant_binomial_ci(
            count,
            total
        )

        participant_metric_records.append({
            "Transfer": transfer,
            "Metric": metric,
            "Participants": total,
            "Events": count,
            "Rate (%)": (
                100 * count / total
                if total > 0 else np.nan
            ),
            "95% CI Lower": ci_lower,
            "95% CI Upper": ci_upper
        })

# ============================================================
# 6. Display participant-level reliability results
# ============================================================

participant_reliability_df = pd.DataFrame(
    participant_metric_records
)

print("\n" + "=" * 70)
print("PARTICIPANT-LEVEL PERFORMANCE")
print("=" * 70)

display(
    participant_reliability_df.round(2)
)

# ============================================================
# 7. Compare recording-level and participant-level detection
# ============================================================

print("\n" + "=" * 70)
print("RECORDING-LEVEL VS PARTICIPANT-LEVEL COMPARISON")
print("=" * 70)

comparison_records = []

for _, row in cross_hospital_results_df.iterrows():

    transfer = f"{row['Source']} -> {row['Target']}"

    patient_row = participant_reliability_df[
        (participant_reliability_df["Transfer"] == transfer)
        &
        (
            participant_reliability_df["Metric"]
            == "All Power-line Clips Detected"
        )
    ].iloc[0]

    comparison_records.append({
        "Transfer": transfer,
        "Clip-Level Sensitivity (%)":
            row["Sensitivity (%)"],
        "Power-line Clips": int(
            row["TP"] + row["FN"]
        ),
        "Participants With Complete Detection (%)":
            patient_row["Rate (%)"],
        "Power-line Participants":
            int(patient_row["Participants"])
    })

comparison_df = pd.DataFrame(comparison_records)

display(
    comparison_df.round(2)
)

# ============================================================
# 8. Final integrity checks
# ============================================================

print("\n" + "=" * 70)
print("PARTICIPANT-LEVEL AUDIT COMPLETE")
print("=" * 70)

print(
    "Total participants represented:",
    patient_df["Patient_ID"].nunique()
)

print(
    "Transfer directions analyzed:",
    patient_df["Transfer"].nunique()
)

print(
    "Detection thresholds modified: No"
)

print(
    "Models retrained: No"
)

print(
    "\nConfidence intervals use participants "
    "as the observational units."
)

print(
    "Estimates remain exploratory because "
    "participant samples are small and selected."
)

print(
    "\nNeuroSignal-QC participant-level "
    "reliability assessment complete."
)

NEUROSIGNAL-QC — PARTICIPANT-LEVEL RELIABILITY

PARTICIPANT-LEVEL PERFORMANCE


,Transfer,Metric,Participants,Events,Rate (%),95% CI Lower,95% CI Upper
0,FNUSA -> Mayo,All Power-line Clips Detected,8,6,75.00,34.91,96.81
1,FNUSA -> Mayo,Any False Alarm,18,3,16.67,3.58,41.42
2,FNUSA -> Mayo,Any Pathological Flag,6,0,0.00,0.00,45.93
3,Mayo -> FNUSA,All Power-line Clips Detected,3,3,100.00,29.24,100.00
4,Mayo -> FNUSA,Any False Alarm,13,0,0.00,0.00,24.71
5,Mayo -> FNUSA,Any Pathological Flag,12,0,0.00,0.00,26.46



RECORDING-LEVEL VS PARTICIPANT-LEVEL COMPARISON


,Transfer,Clip-Level Sensitivity (%),Power-line Clips,Participants With Complete Detection (%),Power-line Participants
0,Mayo -> FNUSA,100.0,15,100.0,3
1,FNUSA -> Mayo,90.0,30,75.0,8



PARTICIPANT-LEVEL AUDIT COMPLETE
Total participants represented: 38
Transfer directions analyzed: 2
Detection thresholds modified: No
Models retrained: No

Confidence intervals use participants as the observational units.
Estimates remain exploratory because participant samples are small and selected.

NeuroSignal-QC participant-level reliability assessment complete.
